# Métricas de avaliação dos modelos

Este notebook define, testa e demonstra as métricas usadas para avaliar os modelos de predição de temperatura.

As funções ficam na célula da Seção 1, que é a fonte canônica do projeto. Ela deve ser copiada sem alteração para os notebooks que avaliarem modelos, de modo que exista uma única definição de métrica em todo o trabalho. Qualquer correção feita aqui precisa ser propagada às cópias.

## As três métricas e o que cada uma mede

### Erro absoluto médio

$$\text{MAE} = \frac{1}{n}\sum_{i=1}^{n} |\hat{y}_i - y_i|$$

Média dos erros em valor absoluto, expressa em graus Celsius. É a métrica mais direta de comunicar à parceira, pois responde à pergunta de quantos graus o modelo erra em média. Todos os erros recebem o mesmo peso, de modo que errar 20 graus uma vez custa o mesmo que errar 2 graus dez vezes.

### Raiz do erro quadrático médio

$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n} (\hat{y}_i - y_i)^2}$$

Também em graus Celsius, e portanto diretamente comparável ao erro absoluto médio. Por elevar ao quadrado antes de somar, penaliza de forma desproporcional os erros grandes. A distinção importa em validação térmica: um erro isolado de 30 graus em um componente é mais grave que dez erros de 3 graus, uma vez que apenas o primeiro leva a uma decisão errada de aprovação.

A razão entre as duas métricas é reportada por constituir diagnóstico gratuito. Valores próximos de 1 indicam erros de magnitude semelhante entre si; valores bem acima de 1 indicam que poucos erros grandes dominam o resultado, situação em que a pergunta seguinte passa a ser quais casos os produzem.

### Coeficiente de determinação

$$R^2 = 1 - \frac{\sum_{i=1}^{n} (\hat{y}_i - y_i)^2}{\sum_{i=1}^{n} (y_i - \bar{y})^2}$$

Adimensional. A leitura correta não é de acurácia, e sim de comparação contra um modelo trivial: o denominador é o erro que se cometeria prevendo sempre a média do alvo. Um valor de 0,85 indica que 85 por cento desse erro foi eliminado. O coeficiente pode ser negativo, o que significa desempenho pior que o da média.

Duas ressalvas acompanham seu uso neste projeto. A primeira é que ele não é comparável entre áreas térmicas, pois seu denominador é a variância do alvo de cada área: uma área de temperaturas homogêneas, como Oven HMI, pode apresentar erro absoluto pequeno e coeficiente próximo de zero, simplesmente por haver pouca variância a explicar. A segunda é que ele é indefinido quando o alvo assume um único valor, caso em que a implementação do scikit-learn devolve zero, valor que se confunde com um resultado legítimo. Por essa razão a função definida adiante devolve um valor ausente nessa situação.

### Erro percentual absoluto médio

$$\text{MAPE} = \frac{100}{n}\sum_{i=1}^{n} \left|\frac{\hat{y}_i - y_i}{y_i}\right|$$

Adimensional, expresso em percentual. É a ponte direta com o indicador acordado com a parceira, que fixa em 10 por cento a margem de erro aceitável, e por isso a métrica de maior peso na conversa de negócio.

Sobre a divisão por zero, a análise é conclusiva e dispensa tratamento defensivo. O denominador é a temperatura absoluta medida, cujo mínimo na base é de 27,69 graus Celsius. Um fogão em operação não produz leitura nula, de modo que o denominador nunca se anula e nenhuma proteção numérica se justifica. O risco real é outro: caso valores padronizados fossem passados por engano, o alvo se aproximaria de zero e o erro percentual explodiria. Por essa razão a validação de entrada recusa alvo não positivo, transformando esse engano em erro visível em lugar de número silenciosamente absurdo.

Cabe registrar uma assimetria da métrica, relevante para este problema. O erro percentual é dividido pelo valor real, o que faz predições baixas serem menos penalizadas que predições altas de igual afastamento em graus. Um preditor constante avaliado sobre alvos de 100 e 200 graus obtém 25 por cento ao prever 100, e 50 por cento ao prever 200, embora ambos errem 100 graus no total. O MAPE, portanto, favorece modelos que subestimam, exatamente o comportamento que o risco R06 identifica como perigoso, uma vez que subestimar temperatura aprova arquiteturas que deveriam ser reprovadas. Por isso ele nunca é lido sozinho, e sim ao lado do viés e das métricas de decisão.

### Erro percentual ponderado

$$\text{WAPE} = 100 \cdot \frac{\sum_{i=1}^{n} |\hat{y}_i - y_i|}{\sum_{i=1}^{n} y_i}$$

Também percentual, mas com um único denominador agregado em lugar de uma divisão por observação. É estruturalmente imune a denominadores pequenos e serve de verificação de robustez: quando as duas métricas concordam, o erro percentual está bem distribuído; quando divergem, os erros estão concentrados nas leituras de temperatura mais baixa.

### Proporção dentro da margem

Percentual de predições cujo erro relativo não excede a margem de 10 por cento. É a leitura literal do indicador do parceiro, e complementa o erro percentual médio: um modelo pode apresentar média aceitável e ainda assim falhar a margem em parte relevante dos casos.

### Métricas de decisão contra o limite de segurança

As métricas anteriores medem distância em graus ou em pontos percentuais. Nenhuma delas informa se o modelo acerta a decisão que a engenharia efetivamente toma, que é aprovar ou reprovar um componente por comparação com o seu limite.

Para os componentes que possuem limite tabelado, a predição e a medição são convertidas em decisão, e a concordância entre as duas é medida. Dos dois tipos de discordância, apenas um é grave. Reprovar um componente que passaria é conservador, e custa uma simulação adicional. Aprovar um componente que na realidade ultrapassa o limite é o erro que a ferramenta existe para evitar, e é contado à parte sob o nome de falso aprovado.

A leitura dessas métricas exige atenção ao número de ensaios que de fato atingem o limite: em um componente que nunca o atinge, a concordância é trivialmente perfeita e nada informa. Por isso essa contagem acompanha as demais.

Esta é a métrica que a Hipótese 3 antecipou, ao concluir que a proximidade do limite importa mais que a temperatura absoluta.

A regra de projeto da engenharia é mais exigente que o limite absoluto: a arquitetura só é aprovada quando a temperatura fica 10 por cento abaixo do limite. Por isso a função devolve também a versão das métricas de decisão sobre essa linha, 0,9 vez o limite, aplicada igualmente ao valor real e ao previsto. O falso aprovado contra a regra é o caso descrito no risco R06: um componente que, na realidade, invade a folga de segurança e que o modelo coloca abaixo dela.

### Métricas auxiliares

O viés é o erro médio com sinal, positivo quando o modelo superestima. Todas as métricas anteriores são de magnitude e ocultam a direção do erro: um modelo que erra dez graus para mais na metade dos casos e dez para menos na outra metade apresenta o mesmo erro absoluto médio de um que erra sempre dez para mais, e os dois são modelos distintos. A distinção importa porque subestimar a temperatura aprova arquiteturas que deveriam ser reprovadas, risco registrado como R06 na Seção 4.1.5.

O número de observações acompanha toda métrica, uma vez que um erro medido sobre sete predições não sustenta a mesma conclusão que outro medido sobre quarenta e seis.

## 1. Definição das funções

Esta é a célula canônica. Ela não depende de arquivo algum e pode ser copiada diretamente para outros notebooks.

A avaliação devolve um dicionário, e não valores soltos, por três razões. A função é chamada para cada componente, para cada dobra, para o baseline e para cada combinação dos experimentos, de modo que chamadas separadas multiplicariam as oportunidades de esquecimento. Uma lista de dicionários se converte em tabela diretamente, que é o formato levado à documentação. E o acréscimo de uma métrica futura não obriga a alterar ponto de chamada algum.

As funções recebem valores em graus Celsius. A conversão de escala, quando existir, é responsabilidade de quem chama, pois misturar as duas responsabilidades é justamente o modo como a conversão acaba esquecida.

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S3/preparacao_testes"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import os
import json

import numpy as np
import pandas as pd


MARGEM_KPI = 0.10   # margem de erro acordada no TAPI
FATOR_REGRA = 0.90  # regra de projeto: aprovar so 10 por cento abaixo do limite (risco R06)


def _valida(y_real, y_previsto):
    """Converte para vetores e recusa entradas que produziriam metrica sem sentido.

    A conversao para numpy e necessaria porque a subtracao entre duas Series do pandas
    alinha pelos indices, e nao pela posicao. Uma Series com indice original e outra
    com indice reiniciado produziriam diferencas entre pares errados, em silencio.

    O alvo precisa ser estritamente positivo. A restricao e legitima neste projeto,
    pois trata-se de temperatura absoluta em graus Celsius, cujo minimo na base e de
    27,69 graus. Ela existe para transformar em erro visivel o engano de passar valores
    padronizados, caso em que o alvo se aproximaria de zero e o erro percentual
    explodiria em vez de acusar o problema.
    """
    y_real = np.asarray(y_real, dtype=float).ravel()
    y_previsto = np.asarray(y_previsto, dtype=float).ravel()

    if y_real.shape != y_previsto.shape:
        raise ValueError(f"dimensoes diferentes: {y_real.shape} e {y_previsto.shape}")
    if y_real.size == 0:
        raise ValueError("nenhuma observacao a avaliar")
    if np.isnan(y_real).any():
        raise ValueError("ha valores ausentes em y_real; a mascara deve ser aplicada antes")
    if np.isnan(y_previsto).any():
        raise ValueError("ha valores ausentes em y_previsto")
    if (y_real <= 0).any():
        raise ValueError(
            "y_real contem valor nao positivo; o erro percentual exige temperatura "
            "absoluta em graus Celsius, e nao valores padronizados")

    return y_real, y_previsto


def avaliar(y_real, y_previsto, limite=None):
    """Metricas de erro de um componente, com o alvo em graus Celsius.

    Sempre devolvidas:

    n            numero de observacoes que sustentam as metricas
    MAE          erro absoluto medio, na unidade do alvo
    RMSE         raiz do erro quadratico medio, na mesma unidade
    R2           fracao da variancia do alvo explicada pelo modelo
    vies         erro medio com sinal, positivo quando o modelo superestima
    RMSE/MAE     razao entre as duas, indicando concentracao dos erros
    MAPE_%       erro percentual absoluto medio
    WAPE_%       erro absoluto total dividido pela soma dos alvos
    dentro_10_%  proporcao de predicoes dentro da margem de 10 por cento do TAPI

    Devolvidas apenas quando um limite de seguranca e informado:

    limite_C          o proprio limite, para conferencia
    n_reprova_real    ensaios em que o componente de fato atinge o limite
    acerto_decisao_%  concordancia entre a decisao real e a prevista
    falso_aprovado    ensaios reprovados na realidade e aprovados pelo modelo
    falso_reprovado   ensaios aprovados na realidade e reprovados pelo modelo
    linha_regra_C         FATOR_REGRA vezes o limite, linha de aprovacao da regra de projeto
    n_reprova_regra_real  ensaios que atingem a linha da regra na realidade
    falso_aprovado_regra  ensaios que atingem a linha na realidade e ficam abaixo dela
                          na predicao; e a aprovacao falsa descrita no risco R06

    O R2 sai ausente quando o alvo assume um unico valor, situacao em que ele e
    indefinido e em que a implementacao do scikit-learn devolve zero, valor que se
    confunde com um resultado legitimo.
    """
    y_real, y_previsto = _valida(y_real, y_previsto)

    erro = y_previsto - y_real
    mae = float(np.mean(np.abs(erro)))

    # A raiz e calculada explicitamente: o parametro squared do mean_squared_error foi
    # removido do scikit-learn na versao 1.9, e o root_mean_squared_error so existe a
    # partir da 1.4. O calculo direto funciona em qualquer versao.
    rmse = float(np.sqrt(np.mean(erro ** 2)))

    if np.unique(y_real).size < 2:
        r2 = float("nan")
    else:
        ss_res = float(np.sum(erro ** 2))
        ss_tot = float(np.sum((y_real - np.mean(y_real)) ** 2))
        r2 = 1.0 - ss_res / ss_tot

    relativo = np.abs(erro) / y_real

    resultado = {
        "n": int(y_real.size),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "vies": float(np.mean(erro)),
        "RMSE/MAE": (rmse / mae) if mae > 0 else float("nan"),
        "MAPE_%": 100.0 * float(np.mean(relativo)),
        "WAPE_%": 100.0 * float(np.sum(np.abs(erro)) / np.sum(y_real)),
        "dentro_10_%": 100.0 * float(np.mean(relativo <= MARGEM_KPI)),
    }

    if limite is not None and np.isfinite(limite):
        reprova_real = y_real >= limite
        reprova_prevista = y_previsto >= limite
        resultado.update({
            "limite_C": float(limite),
            "n_reprova_real": int(reprova_real.sum()),
            "acerto_decisao_%": 100.0 * float(np.mean(reprova_real == reprova_prevista)),
            "falso_aprovado": int(np.sum(reprova_real & ~reprova_prevista)),
            "falso_reprovado": int(np.sum(~reprova_real & reprova_prevista)),
        })

        # Regra de projeto: a mesma linha, 0,9 vez o limite, vale para o real e o previsto
        linha = FATOR_REGRA * float(limite)
        reprova_regra_real = y_real >= linha
        reprova_regra_prevista = y_previsto >= linha
        resultado.update({
            "linha_regra_C": linha,
            "n_reprova_regra_real": int(reprova_regra_real.sum()),
            "falso_aprovado_regra": int(np.sum(reprova_regra_real & ~reprova_regra_prevista)),
        })

    return resultado


def consolidar(tabela, grupo="dataset",
               metricas=("MAE", "RMSE", "R2", "vies", "MAPE_%", "WAPE_%", "dentro_10_%")):
    """Consolida metricas por componente em metricas por area.

    A macro-media atribui o mesmo peso a cada componente e responde se o modelo funciona
    bem em todas as partes do produto. A micro-media pondera pelo numero de observacoes
    e responde qual o erro tipico de uma leitura qualquer.

    O R2 nao e comparavel entre areas, pois seu denominador e a variancia do alvo de
    cada area. Ele indica se o modelo supera a media dentro da propria area.
    """
    if grupo not in tabela.columns:
        raise ValueError(f"coluna de agrupamento ausente: {grupo}")
    if "n" not in tabela.columns:
        raise ValueError("a tabela precisa da coluna n para a micro-media")

    metricas = [m for m in metricas if m in tabela.columns]
    linhas = []

    for nome, bloco in tabela.groupby(grupo):
        linha = {grupo: nome, "componentes": len(bloco), "n_total": int(bloco["n"].sum())}
        for m in metricas:
            valores = bloco[m]
            validos = valores.notna()
            linha[f"{m}_macro"] = (float(valores[validos].mean())
                                   if validos.any() else float("nan"))
            if validos.any() and bloco.loc[validos, "n"].sum() > 0:
                linha[f"{m}_micro"] = float(
                    np.average(valores[validos], weights=bloco.loc[validos, "n"]))
            else:
                linha[f"{m}_micro"] = float("nan")

        # As metricas de decisao sao contagens, e por isso somadas em vez de promediadas
        for contagem in ("n_reprova_real", "falso_aprovado", "falso_reprovado",
                     "n_reprova_regra_real", "falso_aprovado_regra"):
            if contagem in bloco.columns:
                linha[contagem] = int(bloco[contagem].fillna(0).sum())
        linhas.append(linha)

    return pd.DataFrame(linhas).sort_values(grupo).reset_index(drop=True)


print("Metricas definidas: avaliar, consolidar")

## 2. Bateria de testes

Uma função de métrica sempre devolve um número, e por isso nunca falha de modo visível. A verificação precisa vir de casos cuja resposta é conhecida de antemão.

Os casos abaixo cobrem os erros mais comuns de implementação: inversão de sinal, troca entre numerador e denominador, alinhamento por índice em lugar de posição, e o tratamento do coeficiente de determinação quando o alvo é constante.

In [ ]:
testes = []


def caso(nome, condicao, detalhe=""):
    testes.append({"caso": nome, "resultado": "passou" if condicao else "FALHOU",
                   "detalhe": detalhe})
    return condicao


# 1. Predicao perfeita: todo erro e nulo e a variancia e inteiramente explicada
y = np.array([50.0, 80.0, 120.0, 200.0])
m = avaliar(y, y)
caso("predicao perfeita",
     m["MAE"] == 0 and m["RMSE"] == 0 and m["R2"] == 1 and m["vies"] == 0,
     f"MAE={m['MAE']}, R2={m['R2']}")

# 2. Prever a media do proprio conjunto: por definicao o coeficiente e exatamente zero
m = avaliar(y, np.full_like(y, y.mean()))
caso("previsao pela media do conjunto", np.isclose(m["R2"], 0.0), f"R2={m['R2']:.6f}")

# 3. Exemplo calculado a mao: reais 100 e 200, previstos 110 e 180
m = avaliar([100.0, 200.0], [110.0, 180.0])
caso("exemplo conferido manualmente",
     np.isclose(m["MAE"], 15.0) and np.isclose(m["RMSE"], np.sqrt(250.0)),
     f"MAE={m['MAE']:.2f} (esperado 15,00), RMSE={m['RMSE']:.2f} (esperado 15,81)")

# 4. Alvo constante: o coeficiente e indefinido e deve sair como ausente, nao como zero
m = avaliar([5.0, 5.0, 5.0], [5.0, 6.0, 4.0])
caso("alvo constante devolve valor ausente", np.isnan(m["R2"]), f"R2={m['R2']}")

# 5. Subestimacao sistematica: o vies deve ser negativo e igual em modulo ao erro medio
m = avaliar([100.0, 110.0, 120.0], [90.0, 100.0, 110.0])
caso("subestimacao sistematica",
     np.isclose(m["vies"], -10.0) and np.isclose(m["MAE"], 10.0),
     f"vies={m['vies']:.2f}, MAE={m['MAE']:.2f}")

# 6. Alinhamento por posicao, e nao por indice: duas Series com indices distintos
a = pd.Series([100.0, 200.0], index=[10, 11])
b = pd.Series([110.0, 180.0], index=[0, 1])
m = avaliar(a, b)
caso("alinhamento por posicao", np.isclose(m["MAE"], 15.0), f"MAE={m['MAE']:.2f}")

# 7. Entradas invalidas precisam falhar de forma explicita
for descricao, argumentos in [
    ("dimensoes diferentes", ([1.0, 2.0], [1.0])),
    ("conjunto vazio", ([], [])),
    ("valor ausente em y_real", ([1.0, np.nan], [1.0, 2.0])),
]:
    try:
        avaliar(*argumentos)
        caso(f"recusa: {descricao}", False, "nao levantou excecao")
    except ValueError:
        caso(f"recusa: {descricao}", True)


# 8. Erro percentual: reais 100 e 200, previstos 110 e 180, ambos com 10 por cento
m = avaliar([100.0, 200.0], [110.0, 180.0])
caso("erro percentual conferido manualmente",
     np.isclose(m["MAPE_%"], 10.0) and np.isclose(m["WAPE_%"], 10.0),
     f"MAPE={m['MAPE_%']:.2f}%, WAPE={m['WAPE_%']:.2f}%")

# 9. As duas divergem quando os erros se concentram nas leituras baixas
m = avaliar([10.0, 1000.0], [20.0, 1000.0])
caso("WAPE nao e dominado por alvo pequeno",
     np.isclose(m["MAPE_%"], 50.0) and m["WAPE_%"] < 1.0,
     f"MAPE={m['MAPE_%']:.2f}%, WAPE={m['WAPE_%']:.2f}%")

# 10. Proporcao dentro da margem: dois de tres dentro de 10 por cento
m = avaliar([100.0, 100.0, 100.0], [105.0, 115.0, 100.0])
caso("proporcao dentro da margem", np.isclose(m["dentro_10_%"], 200.0 / 3),
     f"dentro_10={m['dentro_10_%']:.1f}%")

# 11. Alvo nao positivo precisa ser recusado, por indicar escala padronizada
try:
    avaliar([0.0, 100.0], [1.0, 100.0])
    caso("recusa: alvo nao positivo", False, "nao levantou excecao")
except ValueError:
    caso("recusa: alvo nao positivo", True)

# 12. Assimetria do erro percentual: prever baixo e menos penalizado que prever alto
baixo = avaliar([100.0, 200.0], [100.0, 100.0])["MAPE_%"]
alto = avaliar([100.0, 200.0], [200.0, 200.0])["MAPE_%"]
caso("assimetria do erro percentual", baixo < alto,
     f"prever 100 -> {baixo:.1f}%, prever 200 -> {alto:.1f}%, com o mesmo erro em graus")

# 13. Decisao contra o limite: o segundo ensaio reprova de verdade e o modelo aprova
m = avaliar([100.0, 120.0], [105.0, 109.0], limite=110.0)
caso("decisao contra o limite",
     m["n_reprova_real"] == 1 and m["falso_aprovado"] == 1
     and m["falso_reprovado"] == 0 and np.isclose(m["acerto_decisao_%"], 50.0),
     f"acerto={m['acerto_decisao_%']:.0f}%, falso aprovado={m['falso_aprovado']}")

# 14. Sem limite informado, as metricas de decisao nao sao produzidas
m = avaliar([100.0, 120.0], [105.0, 109.0])
caso("metricas de decisao ausentes sem limite", "acerto_decisao_%" not in m)

# 15. Regra de projeto: linha em 0,9 x 110 = 99. Os dois ensaios a atingem; o modelo
# preve o primeiro abaixo dela. Pela regra absoluta o falso aprovado e o segundo.
m = avaliar([100.0, 120.0], [98.0, 109.0], limite=110.0)
caso("aprovacao falsa contra a regra de 0,9 x limite",
     np.isclose(m["linha_regra_C"], 99.0) and m["n_reprova_regra_real"] == 2
     and m["falso_aprovado_regra"] == 1 and m["falso_aprovado"] == 1,
     f"linha={m['linha_regra_C']:.1f}, falso aprovado pela regra={m['falso_aprovado_regra']}")

testes = pd.DataFrame(testes)
print(testes.to_string(index=False))
print()
assert (testes["resultado"] == "passou").all(), "ha caso de teste reprovado"
print(f"{len(testes)} casos de teste aprovados.")

## 3. Demonstração sobre os dados reais

As métricas são aplicadas a dois preditores, sobre a partição gravada e as dobras de validação nela registradas.

O primeiro é o baseline, que prevê para cada componente a média histórica daquele componente, calculada apenas sobre as configurações de treino de cada dobra. Ele representa a regra que um engenheiro aplicaria à mão hoje, e é contra ele que o modelo precisa demonstrar ganho.

O segundo é o Ridge, primeiro modelo candidato, cujo protocolo está descrito no notebook de modelagem.

Sem o baseline, o coeficiente de determinação não teria referencial concreto, uma vez que a Hipótese 1 já demonstrou que a área do produto explica boa parte da variância da temperatura.

In [ ]:
PASTA_DADOS = "colabs_S4"
PASTA_PARTICAO = "preparacao_testes"
SUFIXO = "_13cat"

PREFIXOS = ("blower_direction_", "housing_", "board_position_", "run_case_",
            "losses_", "perfil_perdas_", "deflector_hole_", "material_coil_support_", "cfg_",
          "reason_", "loss_profile_order_")
AUXILIARES = ["run_idx"]

BASES = [".", "..", os.path.join("..", ".."), "/content",
         "notebooks", os.path.join("..", "notebooks"),
         PASTA_PARTICAO, os.path.join("..", PASTA_PARTICAO),
         os.path.join("notebooks", PASTA_PARTICAO)]


def resolve(pasta, arquivo):
    """Localiza um arquivo com ou sem a pasta que o contem."""
    tentativas = []
    # A pasta declarada tem prioridade sobre o arquivo solto: sem isso, um
    # homonimo numa pasta vizinha pode ser lido no lugar do arquivo pedido.
    for partes in ((pasta, arquivo), (arquivo,)):
        for base in BASES:
            caminho = os.path.join(base, *partes)
            tentativas.append(caminho)
            if os.path.exists(caminho):
                return caminho
    raise FileNotFoundError("Arquivo nao encontrado: " + arquivo
                            + ". Diretorio atual: " + os.path.abspath(".")
                            + ". Caminhos tentados: " + "; ".join(tentativas))


with open(resolve(PASTA_PARTICAO, "particao_treino_teste.json"), encoding="utf-8") as f:
    particao = json.load(f)

DOBRA = {int(k): v for k, v in particao["dobra_por_run_idx"].items()}
DOBRAS = sorted({d for d in DOBRA.values() if d is not None})

pasta = os.path.dirname(resolve(PASTA_DADOS, f"dataset_mainback{SUFIXO}.csv"))
areas = {}
for nome in ["coil_temps", "ct_knobs", "ipc_comp", "ipc_cooling",
             "main_vent_sys", "mainback", "oven_hmi"]:
    areas[nome] = pd.read_csv(os.path.join(pasta, f"dataset_{nome}{SUFIXO}.csv"))

limites = pd.read_csv(resolve(PASTA_DADOS, "limites_ativos.csv"))
LIMITE = limites.set_index("tc_name")["limite_num"].to_dict()

print("Areas carregadas:", len(areas))
print("Componentes com limite tabelado:",
      sum(1 for v in LIMITE.values() if pd.notna(v)))
print("Dobras de validacao:", len(DOBRAS))

In [ ]:
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler

ALPHAS = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
MIN_TREINO, MIN_PREDICOES = 8, 5


def separar(df):
    entrada = [c for c in df.columns if c.startswith(PREFIXOS)]
    alvo = [c for c in df.columns if c not in entrada + AUXILIARES]
    return df[entrada], alvo


def indices_da_dobra(df, dobra, y_col):
    run = df["run_idx"].to_numpy() if "run_idx" in df.columns else np.arange(len(df))
    medido = y_col.notna().to_numpy()
    treino = np.array([DOBRA.get(r) is not None for r in run])
    val = [i for i in range(len(df)) if medido[i] and treino[i] and DOBRA[run[i]] == dobra]
    tr = [i for i in range(len(df)) if medido[i] and treino[i] and DOBRA[run[i]] != dobra]
    return tr, val


linhas = []
for nome, df in sorted(areas.items()):
    X, alvos = separar(df)

    for col in alvos:
        y = df[col]
        if y.notna().sum() < MIN_PREDICOES:
            continue

        reais, pred_base, pred_ridge = [], [], []

        for d in DOBRAS:
            tr, val = indices_da_dobra(df, d, y)
            if len(tr) < MIN_TREINO or not val:
                continue

            y_tr, y_val = y.iloc[tr], y.iloc[val]

            # Baseline: media do componente no treino da propria dobra
            pred_base.extend([float(y_tr.mean())] * len(val))

            escala = StandardScaler()
            modelo = RidgeCV(alphas=ALPHAS)
            modelo.fit(escala.fit_transform(X.iloc[tr]), y_tr)
            pred_ridge.extend(modelo.predict(escala.transform(X.iloc[val])).tolist())

            reais.extend(y_val.tolist())

        if len(reais) < MIN_PREDICOES:
            continue

        for rotulo, previsto in [("baseline", pred_base), ("ridge", pred_ridge)]:
            lim = LIMITE.get(col)
            lim = float(lim) if lim is not None and pd.notna(lim) else None
            linhas.append({"dataset": nome, "componente": col, "preditor": rotulo,
                           **avaliar(reais, previsto, limite=lim)})

resultados = pd.DataFrame(linhas)
print(f"Componentes avaliados: {resultados['componente'].nunique()}")
print(f"Linhas na tabela: {len(resultados)} (um par de preditores por componente)")
print()
print(resultados.head(6).round(3).to_string(index=False))

## 4. Consolidação por área

As métricas por componente são consolidadas por área térmica em duas formas de agregação, que respondem a perguntas diferentes.

A macro-média atribui o mesmo peso a cada componente e informa se o modelo funciona bem em todas as partes do produto. A micro-média pondera pelo número de observações e informa o erro típico de uma leitura qualquer, ficando dominada pelas áreas com mais componentes medidos. Como o IPC Comp reúne 37 das 74 colunas-alvo, as duas contas divergem de forma apreciável, e por isso ambas são apresentadas.

In [ ]:
for rotulo in ["baseline", "ridge"]:
    bloco = resultados[resultados["preditor"] == rotulo]
    print(f"--- {rotulo} ---")
    print(consolidar(bloco)[
        ["dataset", "componentes", "n_total", "MAE_macro", "MAE_micro", "R2_macro"]
    ].round(3).to_string(index=False))
    print()

## 5. Erro percentual e o indicador de 10 por cento

O indicador acordado com a parceira fixa em 10 por cento a margem de erro aceitável. A tabela abaixo confronta o erro percentual médio de cada área com esse valor, e acrescenta a proporção de predições que efetivamente respeitam a margem.

As duas leituras não coincidem, e a diferença entre elas é informativa: um modelo pode apresentar erro percentual médio dentro da margem e ainda assim violá-la em parte relevante dos casos, o que ocorre quando poucos erros grandes convivem com muitos erros pequenos.

In [ ]:
kpi = consolidar(resultados[resultados["preditor"] == "ridge"])[
    ["dataset", "componentes", "MAPE_%_macro", "WAPE_%_macro", "dentro_10_%_macro"]]
kpi.columns = ["dataset", "componentes", "MAPE %", "WAPE %", "dentro da margem %"]
kpi["atende o indicador"] = np.where(kpi["MAPE %"] <= 100 * MARGEM_KPI, "sim", "nao")

print("Ridge, erro percentual por area:")
print(kpi.round(2).to_string(index=False))
print()
print("O WAPE proximo do MAPE indica erro percentual bem distribuido entre as leituras;")
print("divergencia entre os dois apontaria concentracao nas temperaturas mais baixas.")

## 6. Métricas de decisão contra o limite de segurança

As métricas anteriores medem distância. Esta mede se o modelo acerta a decisão que a engenharia toma, que é aprovar ou reprovar o componente por comparação com o seu limite tabelado.

A contagem de falsos aprovados é a que importa, pois corresponde ao componente que ultrapassa o limite na realidade e é aprovado pelo modelo. Reprovar indevidamente é conservador e custa uma simulação; aprovar indevidamente é o erro que a ferramenta existe para evitar.

A leitura precisa considerar quantos ensaios de fato atingem o limite: onde não há nenhum, a concordância é trivialmente perfeita e nada informa.

In [ ]:
decisao = resultados[(resultados["preditor"] == "ridge")
                     & resultados["limite_C"].notna()].copy()

print(f"Componentes com limite tabelado: {len(decisao)} de "
      f"{resultados['componente'].nunique()}")
print(f"Componentes que atingem o limite em algum ensaio: "
      f"{int((decisao['n_reprova_real'] > 0).sum())}")
print()

por_area = decisao.groupby("dataset").agg(
    componentes=("componente", "count"),
    ensaios_que_reprovam=("n_reprova_real", "sum"),
    falsos_aprovados=("falso_aprovado", "sum"),
    falsos_reprovados=("falso_reprovado", "sum"),
    acerto_medio=("acerto_decisao_%", "mean"),
).reset_index()

print("Decisao de aprovacao por area, modelo Ridge:")
print(por_area.round(1).to_string(index=False))
print()

criticos = decisao[decisao["falso_aprovado"] > 0].nlargest(8, "falso_aprovado")
if len(criticos):
    print("Componentes com maior numero de falsos aprovados:")
    print(criticos[["dataset", "componente", "limite_C", "n_reprova_real",
                    "falso_aprovado", "acerto_decisao_%"]].round(1).to_string(index=False))
else:
    print("Nenhum falso aprovado registrado.")

## 7. Interpretação frente ao piso de ruído

O erro de cada área é comparado com o ruído experimental da própria área, medido pela dispersão dos alvos entre repetições da mesma configuração e registrado na Seção 4.3.1. Esse valor é um limite inferior: nenhum modelo pode ficar abaixo dele, uma vez que dois ensaios da mesma configuração já discordam nessa magnitude.

A comparação com o baseline responde à pergunta complementar, que é se o modelo aprende algo além do patamar histórico de cada componente.

In [ ]:
# Piso de ruido medido sobre a base atual, com a chave de replicata e o
# estimador agrupado. O calculo vive na Secao 7 do notebook de modelagem, que o
# refaz a partir dos dados; os valores sao repetidos aqui por este notebook nao
# carregar os conjuntos completos. Substituem os valores anteriores, medidos
# sobre a chave de sete variaveis e sobre versao anterior da base.
# Piso de ruido por area: raiz da variancia agrupada dentro dos grupos de
# repeticao da chave de replicata (notebooks/colabs_S4/chave_replicata.json),
# ponderada por graus de liberdade. Cinco pares; o par formado pelos pontos 7 e 8 do
# planejamento de posicao do soprador deixou de contar como repeticao, porque a
# posicao nao consta de nenhuma coluna. Valores conferidos em analise_dataset.
RUIDO_EXP = {
    "oven_hmi": 0.63, "ct_knobs": 0.92, "main_vent_sys": 1.19, "ipc_cooling": 0.87,
    "mainback": 0.57, "ipc_comp": 3.52, "coil_temps": 5.07,
}

comp = consolidar(resultados[resultados["preditor"] == "ridge"])[["dataset", "MAE_macro"]]
comp = comp.rename(columns={"MAE_macro": "ridge"})
base = consolidar(resultados[resultados["preditor"] == "baseline"])[["dataset", "MAE_macro"]]
comp = comp.merge(base.rename(columns={"MAE_macro": "baseline"}), on="dataset")
comp["ruido"] = comp["dataset"].map(RUIDO_EXP)
comp["ridge/ruido"] = comp["ridge"] / comp["ruido"]
comp["ganho sobre baseline %"] = 100 * (comp["baseline"] - comp["ridge"]) / comp["baseline"]

print(comp.round(2).to_string(index=False))
print()
print("Todos os valores em graus Celsius, exceto as duas ultimas colunas.")

## 8. Registro

A célula da Seção 1 é a definição canônica das métricas do projeto. Ela deve substituir o cálculo embutido no notebook de modelagem, de modo que exista uma única implementação, e precisa ser copiada sem alteração. Toda correção feita aqui exige propagação às cópias, e é essa a razão de a célula ser autocontida e não depender de arquivo externo.

O acréscimo do erro percentual e das métricas de decisão contra o limite de segurança está previsto em card próprio, e não altera nenhum ponto de chamada, uma vez que a função devolve um dicionário.